In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/imaterialist-challenge-furniture-2018'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, shutil
import random
import json
import pathlib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from multiprocessing.pool import ThreadPool as Pool 

import matplotlib.pyplot as plt
import seaborn as sns
import urllib3

import torch
from torch import nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

import torchvision
from torchvision import transforms, models
from torchvision.utils import make_grid
from torchvision.datasets import ImageFolder
from torchvision.io import read_image
from torchvision.transforms import ToTensor, Lambda


%matplotlib inline

In [ ]:
class Config:
    # Training
    epochs = 10
    learning_rate = 0.001
    num_classes = 129
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    
    # Data preparation
    train_data_root = '/kaggle/input/imaterialist-challenge-furniture-2018/train.json'
    test_data_root = '/kaggle/input/imaterialist-challenge-furniture-2018/test.json'
    validation_data_root = '/kaggle/input/imaterialist-challenge-furniture-2018/validation.json'
    batch_size = 16
    num_workers = 2
    anotation_file_root = '/kaggle/input/imaterialist-furniture-fgvc5/{}.csv'
    images_path = '/kaggle/input/imaterialist-furniture-fgvc5/images/{}'
    test_anotation_file_root = '/kaggle/working/test_df.csv'
    
    # Data transformation
    mean = (0.485, 0.456, 0.406) 
    std = (0.229, 0.224, 0.225) 
    resize_to = 256
    img_size = 224

In [ ]:
train_data = pd.read_json(Config.train_data_root)
test_data = pd.read_json(Config.test_data_root)
valid_data = pd.read_json(Config.validation_data_root)

print(test_data.size)

In [ ]:
def format_dataset(data, test_data=False):
    df = pd.DataFrame()
    if test_data:
        df['image_id'] = data.images.map(lambda x: x['image_id'])
        df['url'] = data.images.map(lambda x: x['url'][0])
    else:
        df['image_id'] = data.annotations.map(lambda x: x['image_id'])
        df['label_id'] = data.annotations.map(lambda x: x['label_id'])
        df['url'] = data.images.map(lambda x: x['url'])
    return df

train_df = format_dataset(train_data)
valid_df = format_dataset(valid_data)
test_df = format_dataset(test_data, test_data=True)



In [ ]:
print(test_df.shape)

In [ ]:
test_df.to_csv('test_df.csv', index=False)

In [ ]:
class_count = train_df.groupby(by=['label_id']).size()
print(class_count)

plt.figure(figsize = (10, 8))
plt.title('Category Distribuition')
sns.distplot(train_df['label_id'])

plt.show()

In [ ]:
# !cd /kaggle/working/
# !rm -r test
# !mkdir test
# !mkdir train
# !mkdir validation

In [ ]:
def get_transforms(data_subset):
    if data_subset == 'train':
        transformations = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip(),
            transforms.RandomResizedCrop((Config.img_size, Config.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(Config.mean, Config.std)
        ])
    else:
        transformations = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((Config.img_size, Config.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(Config.mean, Config.std)
        ])
    return transformations

In [ ]:
print(train_df.head())

In [ ]:
print(train_df.iloc[4, 1])

In [ ]:
class CustomImageDataset(TensorDataset):
    def __init__(self, annotations_file, img_dir, transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.shuffle()

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f'{self.img_labels.iloc[idx, 0]}.jpg')
        image = read_image(img_path)
        image = self.transform(image)
        label = self.img_labels.iloc[idx, 1]
        return image, label

    def shuffle(self):
        self.img_labels = self.img_labels.sample(frac=1).reset_index(drop=True)

In [ ]:
class TestImageDataset(TensorDataset):
    def __init__(self, annotations_file, img_dir, transform=None):
        self.img_ids = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        image_id = self.img_ids.iloc[idx, 0]
        img_path = os.path.join(self.img_dir, f'{self.img_ids.iloc[idx, 0]}.jpg')
        try:
            image = read_image(img_path)
            image = self.transform(image)
            return image, image_id
        except:
            return [], image_id

In [ ]:
print(test_df.shape)

In [ ]:
dfs = {
    'test': test_df,
    'train': train_df,
    'validation': valid_df
}

In [ ]:
dataloaders = {}
image_datasets = {}
for name in ['train', 'validation', 'test']:
    if name == 'test':
        image_datasets[name] = TestImageDataset(Config.test_anotation_file_root, 
                                     Config.images_path.format(name),
                                     transform=get_transforms(name))
        dataloaders[name] = DataLoader(image_datasets[name], 
                                       shuffle=False, 
                                       num_workers=0)
    else:
        image_datasets[name] = CustomImageDataset(Config.anotation_file_root.format(name), 
                                     Config.images_path.format(name),
                                     transform=get_transforms(name))
        dataloaders[name] = DataLoader(image_datasets[name], 
                                       shuffle=True, 
                                       batch_size=Config.batch_size,
                                       num_workers=0)

In [ ]:
def tensor_to_array(img, mean = Config.mean, std = Config.std):
    img = img.numpy()
    np_img = np.transpose(img, (1, 2, 0))
    np_img = np.multiply(np_img, Config.std) + Config.mean
    np_img = np.clip(np_img, 0, 1)

    return np_img

In [ ]:
def show_grid(grid, title=None):
    """
    displays a grid of images
    """
    np_grid = tensor_to_array(grid)
    
    # display
    plt.figure(figsize=(10, 5))
    plt.axis('off')
    plt.imshow(np_grid)
    if title is not None:
        plt.title(title) 

In [ ]:
for name in ['train', 'validation', 'test']:
    for imgs, _ in dataloaders[name]:
        if imgs != []:
            out = make_grid(imgs)
            show_grid(out, f'{name} samples')
            break
        else:
            continue

In [ ]:
# class DensNet(nn.Module):
#     def __init__(self, num_classes=1000, num_channels=3):
#         super().__init__()
#         preloaded = torchvision.models.densenet121(pretrained=True)
#         self.features = preloaded.features
#         self.features.conv0 = nn.Conv2d(num_channels, 64, 7, 2, 3)
#         self.classifier = nn.Linear(1024, num_classes, bias=True)
#         del preloaded
        
#     def forward(self, x):
#         features = self.features(x)
#         out = F.relu(features, inplace=True)
#         out = F.adaptive_avg_pool2d(out, (1, 1)).view(features.size(0), -1)
#         out = self.classifier(out)
#         return out

In [ ]:
# model = DensNet(num_classes=Config.num_classes).to(Config.device)

In [ ]:
model = models.resnet18(pretrained=True).to(Config.device)
    
for param in model.parameters():
    param.requires_grad = False   
    
output_features = model.fc.in_features
    
model.fc = nn.Linear(output_features, Config.num_classes).to(Config.device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=Config.learning_rate, weight_decay=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=3):
    callbackers = {}
    callbackers['train_loss'] = []
    callbackers['validation_loss'] = []
    callbackers['train_accuracy'] = []
    callbackers['validation_accuracy'] = []
    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch+1, num_epochs))
        print('-' * 10)

        for phase in ['train', 'validation']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(Config.device)
                labels = labels.to(Config.device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                _, preds = torch.max(outputs, 1)
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print('{} loss: {:.4f}, acc: {:.4f}'.format(phase,
                                                        epoch_loss,
                                                        epoch_acc))
            callbackers[f'{phase}_loss'].append(epoch_loss)
            callbackers[f'{phase}_accuracy'].append(epoch_acc)
            
    return model, callbackers

In [ ]:
model_trained, callbackers = train_model(model, criterion, optimizer, num_epochs=Config.epochs)

In [ ]:
def visualize_callbackers(callbackers):
    fig, ax =  plt.subplots(2, 1, figsize=(10, 6))
    ax[0].plot(callbackers['train_loss'], label='Train Loss')
    ax[0].plot(callbackers['validation_loss'], label='Valid Loss')
    ax[0].set_title('Losses')
    ax[0].legend()
    ax[1].plot(callbackers['train_accuracy'], label='Train Accuracy')
    ax[1].plot(callbackers['validation_accuracy'], label='Valid Accuracy')
    ax[1].set_title('Classification metrics')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# visualize_callbackers(callbackers)

In [ ]:
!cd /kaggle/working/
!mkdir models
!mkdir models/pytorch

In [ ]:
torch.save(model_trained.state_dict(), '/kaggle/working/models/pytorch/weights.h5')

In [ ]:
model = models.resnet18(pretrained=False).to(Config.device)
model.fc = nn.Linear(output_features, Config.num_classes).to(Config.device)
model.load_state_dict(torch.load('/kaggle/working/models/pytorch/weights.h5'))

In [ ]:
# model = DensNet(num_classes=Config.num_classes)
# model.to(Config.device)
# model.load_state_dict(torch.load('/kaggle/working/models/pytorch/weights.h5'))

In [ ]:
@torch.no_grad()
def prediction(model, loader):
    df = pd.DataFrame({'id': pd.Series(dtype='int'),
                      'predicted': pd.Series(dtype='int')})
    for img, img_id in loader:
        if img == []:
            prediction = 42
        else:
            img_id = img_id.to(Config.device)
            img = img.to(Config.device)
            outputs = model(img)
            preds = outputs.max(dim=-1)[1].cpu()
            img_id = img_id.cpu()
            prediction = preds.detach().numpy()[0]
        subm = {'id': img_id.item(), 
                'predicted': prediction}
        df = df.append(subm, ignore_index=True)
    return df

In [ ]:
submission = prediction(model, dataloaders['test'])

In [ ]:
submission.to_csv('submission.csv', index=False)